# ViTox Pixel Lab — Gradio demo

Notebook này tạo ứng dụng Gradio từ hai Kaggle Inputs:

- **NB00**: `vihsd_manifest.json` và `validation.parquet`.
- **Checkpoint seed 42**: ưu tiên output `Checkpoints_TMG` hoặc NB03 tại `seed_42/best_model`; cũng chấp nhận `original/best_model` từ output augmentation trong ảnh nếu metadata xác nhận `seed=42`, `variant=original`, `augmentation=none`.

Ứng dụng khóa cứng **seed 42**, ghép prediction với validation sau khi xác minh thứ tự nhãn, rồi chọn đúng **2 CLEAN + 2 OFFENSIVE + 2 HATE** mà model dự đoán đúng. Sáu ví dụ là mẫu được tuyển chọn để demo, không phải một phép đánh giá độc lập.

Tên Kaggle Dataset/Input khuyến nghị: `tmg-toxic-nb03-seed42`. Artifact tối thiểu gồm `best_model/`, `predictions_validation.parquet` và `validation_result.json`; không cần upload thư mục recovery `checkpoints/` để chạy app.

> ViHSD là dữ liệu gated. Giữ `SHARE_APP = False` khi chưa xác minh quyền công bố nguyên văn các mẫu dữ liệu.


In [ ]:
# ===== USER SETTINGS — run this cell first =====
SHARE_APP = False  # Chỉ đổi thành True sau khi xác minh quyền chia sẻ câu ViHSD.
TARGET_SEED = 42
TARGET_VARIANT = 'original'
MAX_LENGTH = 128
SAMPLES_PER_CLASS = {0: 2, 1: 2, 2: 2}

import os, sys, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Không cài lại torch/CUDA của Kaggle.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gradio>=5.0,<6.0', 'transformers==4.57.1', 'sentencepiece',
    'protobuf', 'safetensors', 'pandas', 'pyarrow',
], check=True)
print({'share_app': SHARE_APP, 'target_seed': TARGET_SEED,
       'target_variant': TARGET_VARIANT, 'samples_per_class': SAMPLES_PER_CLASS})


In [ ]:
from pathlib import Path
import html, json, time
import numpy as np
import pandas as pd
import torch

KAGGLE_INPUT = Path('/kaggle/input')
LABEL2ID = {'CLEAN': 0, 'OFFENSIVE': 1, 'HATE': 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

def require_exactly_one(pattern):
    matches = list(KAGGLE_INPUT.rglob(pattern))
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly one {pattern!r} under /kaggle/input; found {matches}'
        )
    return matches[0]

# NB00 — prepared ViHSD validation data.
manifest_path = require_exactly_one('vihsd_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('artifact') != 'vihsd_prepared_v1':
    raise ValueError('Unexpected prepared-data artifact.')
validation_spec = manifest['splits']['validation']
validation_path = manifest_path.parent / validation_spec['file']
validation_frame = pd.read_parquet(validation_path).reset_index(drop=True)
if len(validation_frame) != int(validation_spec['rows']):
    raise ValueError('NB00 validation row count differs from its manifest.')

# Resolve exactly the unaugmented seed-42 artifact. NB03 layout has priority.
def resolve_seed_artifact(seed, variant):
    candidates = []
    for result_path in KAGGLE_INPUT.rglob('validation_result.json'):
        try:
            payload = json.loads(result_path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        if payload.get('seed') != seed:
            continue
        if payload.get('variant') != variant or payload.get('augmentation') != 'none':
            continue
        artifact_dir = result_path.parent
        model_path = artifact_dir / 'best_model'
        prediction_path = artifact_dir / 'predictions_validation.parquet'
        if not model_path.is_dir() or not prediction_path.is_file():
            continue
        # Full NB03 output: seed_42. Screenshot/NB04 fallback: original.
        rank = 0 if artifact_dir.name == f'seed_{seed}' else (
            1 if artifact_dir.name == variant else 2
        )
        candidates.append((rank, str(result_path), result_path, payload))
    if not candidates:
        raise RuntimeError(
            'Cannot find a complete seed-42 original artifact. Expected either '
            'seed_42/{best_model,predictions_validation.parquet,validation_result.json} '
            'or original/{best_model,predictions_validation.parquet,validation_result.json}.'
        )
    candidates.sort(key=lambda item: (item[0], item[1]))
    best_rank = candidates[0][0]
    preferred = [item for item in candidates if item[0] == best_rank]
    if len(preferred) != 1:
        raise RuntimeError(
            f'Ambiguous seed-42 artifacts: {[str(item[2]) for item in preferred]}'
        )
    _, _, result_path, payload = preferred[0]
    return result_path.parent, result_path, payload

seed_dir, validation_result_path, validation_result = resolve_seed_artifact(
    TARGET_SEED, TARGET_VARIANT
)
best_seed = TARGET_SEED
model_dir = seed_dir / 'best_model'
predictions_path = seed_dir / 'predictions_validation.parquet'
best_validation_macro_f1 = float(
    validation_result['raw_validation_metrics']['macro_f1']
)
experiment_name = str(validation_result.get('experiment', 'UNKNOWN'))
artifact_origin = (
    'NB03 seed_42' if seed_dir.name == 'seed_42'
    else 'NB04 original seed42' if seed_dir.name == 'original'
    else experiment_name
)
config_path = model_dir / 'config.json'
if not config_path.is_file():
    raise FileNotFoundError(config_path)
model_config = json.loads(config_path.read_text(encoding='utf-8'))
if model_config.get('model_type') != 'visobert-optimized-gate-v4':
    raise ValueError('Attached best_model is not Optimized GATED v4.')
weight_candidates = (
    model_dir / 'model.safetensors', model_dir / 'pytorch_model.bin',
    model_dir / 'model.safetensors.index.json',
)
if not any(path.is_file() for path in weight_candidates):
    raise FileNotFoundError('best_model has no supported model weights.')
for tokenizer_file in ('tokenizer_config.json', 'sentencepiece.bpe.model'):
    tokenizer_path = model_dir / tokenizer_file
    if not tokenizer_path.is_file():
        raise FileNotFoundError(tokenizer_path)

print({
    'manifest': str(manifest_path),
    'validation_rows': len(validation_frame),
    'selected_seed': best_seed,
    'artifact_origin': artifact_origin,
    'validation_result': str(validation_result_path),
    'raw_validation_macro_f1': round(best_validation_macro_f1, 6),
    'model_dir': str(model_dir),
    'weights': [path.name for path in weight_candidates if path.is_file()],
})


## Custom Optimized GATED v4 architecture

NB03 lưu đủ weights/config/tokenizer nhưng đây là custom Transformers architecture. Các class dưới đây phải tồn tại trước khi gọi `from_pretrained`.


In [ ]:
from torch import nn
import torch.nn.functional as F
from transformers import (
    AutoConfig, AutoModel, AutoTokenizer, PretrainedConfig, PreTrainedModel,
)
from transformers.modeling_outputs import SequenceClassifierOutput

class OptimizedGateConfig(PretrainedConfig):
    model_type, is_composition = 'visobert-optimized-gate-v4', True

    def __init__(self, encoder_config=None, encoder_model_id='uitnlp/visobert',
                 encoder_revision=None, last_n_layers=4, cnn_dim=192,
                 cnn_kernels=(3, 5, 7), fusion_dim=256, cnn_dropout=.1,
                 head_dropout=.15, class_weights=None, focal_gamma=.5,
                 aux_toxic_weight=.2, aux_target_weight=.25, toxic_pos_weight=1.,
                 gate_scale_logit_init=-1.5, content_special_ids=None,
                 num_labels=3, **kwargs):
        kernels = tuple(int(value) for value in cnn_kernels)
        if last_n_layers < 1 or any(value < 1 or value % 2 == 0 for value in kernels):
            raise ValueError('Invalid layer count or CNN kernels.')
        super().__init__(num_labels=num_labels, **kwargs)
        self.encoder_config = dict(encoder_config or {})
        self.encoder_model_id, self.encoder_revision = encoder_model_id, encoder_revision
        self.last_n_layers, self.cnn_dim = int(last_n_layers), int(cnn_dim)
        self.cnn_kernels, self.fusion_dim = list(kernels), int(fusion_dim)
        self.cnn_dropout, self.head_dropout = float(cnn_dropout), float(head_dropout)
        self.class_weights = list(class_weights or [])
        self.focal_gamma = float(focal_gamma)
        self.aux_toxic_weight = float(aux_toxic_weight)
        self.aux_target_weight = float(aux_target_weight)
        self.toxic_pos_weight = float(toxic_pos_weight)
        self.gate_scale_logit_init = float(gate_scale_logit_init)
        self.content_special_ids = list(content_special_ids or [])

class ResidualDsConv(nn.Module):
    def __init__(self, channels, kernel, dropout):
        super().__init__()
        self.depthwise = nn.Conv1d(channels, channels, kernel, padding=kernel // 2,
            groups=channels, bias=False)
        self.pointwise = nn.Conv1d(channels, channels, 1)
        self.dropout, self.norm = nn.Dropout(dropout), nn.LayerNorm(channels)

    def forward(self, tokens, mask):
        valid = mask.unsqueeze(-1).to(tokens.dtype)
        tokens = tokens * valid
        update = self.pointwise(F.gelu(
            self.depthwise(tokens.transpose(1, 2)))).transpose(1, 2)
        return self.norm(tokens + self.dropout(update)) * valid

class MultiScaleDsCnnPool(nn.Module):
    def __init__(self, hidden, channels, kernels, dropout):
        super().__init__()
        self.projection = nn.Linear(hidden, channels)
        self.norm = nn.LayerNorm(channels)
        self.branches = nn.ModuleList([ResidualDsConv(channels, kernel, dropout)
            for kernel in kernels])
        self.output_size = 2 * channels * len(kernels)

    def forward(self, hidden, mask):
        if hidden.shape[:2] != mask.shape or not bool(mask.any(dim=1).all()):
            raise ValueError('Invalid hidden/mask pair.')
        valid = mask.unsqueeze(-1).to(hidden.dtype)
        tokens = self.norm(F.gelu(self.projection(hidden))) * valid
        pooled = []
        for branch in self.branches:
            features = branch(tokens, mask)
            maximum = features.masked_fill(~mask.unsqueeze(-1),
                torch.finfo(features.dtype).min).amax(dim=1)
            mean = (features * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
            pooled.extend((maximum, mean))
        return torch.cat(pooled, dim=-1)

def relation_features(left, right):
    return torch.cat([left, right, torch.abs(left - right), left * right], dim=-1)

def binary_head(input_dim, hidden_dim, dropout):
    return nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.GELU(),
        nn.LayerNorm(hidden_dim), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

class OptimizedGateViSoBERT(PreTrainedModel):
    config_class, base_model_prefix = OptimizedGateConfig, 'encoder'
    main_input_name, accepts_loss_kwargs = 'input_ids', False

    def __init__(self, config):
        super().__init__(config)
        payload = dict(config.encoder_config)
        model_type = payload.pop('model_type', None)
        if not model_type:
            raise ValueError('encoder_config.model_type is required')
        encoder_config = AutoConfig.for_model(model_type, **payload)
        encoder_config._attn_implementation = 'eager'
        if config.last_n_layers > int(encoder_config.num_hidden_layers):
            raise ValueError('last_n_layers exceeds encoder depth')
        hidden, dim = int(encoder_config.hidden_size), config.fusion_dim
        self.encoder = AutoModel.from_config(encoder_config, add_pooling_layer=False)
        self.layer_weights = nn.Parameter(torch.zeros(config.last_n_layers))
        self.global_projection = nn.Linear(2 * hidden, dim)
        self.global_norm = nn.LayerNorm(dim)
        self.local_pool = MultiScaleDsCnnPool(hidden, config.cnn_dim,
            config.cnn_kernels, config.cnn_dropout)
        self.local_projection = nn.Linear(self.local_pool.output_size, dim)
        self.local_norm = nn.LayerNorm(dim)
        self.gate_hidden = nn.Linear(4 * dim, dim)
        self.gate_output = nn.Linear(dim, dim)
        self.residual_scale_logit = nn.Parameter(
            torch.tensor(config.gate_scale_logit_init))
        self.fusion_norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(config.head_dropout)
        self.classifier = nn.Linear(dim, config.num_labels)
        self.toxic_head = binary_head(4 * dim, dim, config.head_dropout)
        self.target_head = binary_head(4 * dim, dim, config.head_dropout)
        self.register_buffer('class_weight_tensor',
            torch.tensor(config.class_weights, dtype=torch.float32), persistent=False)
        self.register_buffer('toxic_pos_weight_tensor',
            torch.tensor(config.toxic_pos_weight, dtype=torch.float32), persistent=False)
        self.post_init()
        nn.init.zeros_(self.gate_output.weight)
        nn.init.zeros_(self.gate_output.bias)

    def content_mask(self, input_ids, attention_mask):
        mask = attention_mask.bool().clone()
        for token_id in self.config.content_special_ids:
            mask &= input_ids.ne(int(token_id))
        empty = ~mask.any(dim=1)
        if empty.any():
            mask[empty] = attention_mask[empty].bool()
        return mask

    def compute_features(self, input_ids, attention_mask, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True, return_dict=True, **kwargs)
        selected = torch.stack(outputs.hidden_states[-self.config.last_n_layers:], dim=0)
        mixing = self.layer_weights.softmax(dim=0).view(-1, 1, 1, 1)
        hidden = (selected * mixing).sum(dim=0)
        mask = self.content_mask(input_ids, attention_mask)
        valid = mask.unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
        global_vector = self.global_norm(F.gelu(self.global_projection(
            torch.cat([hidden[:, 0], mean], dim=-1))))
        local_vector = self.local_norm(F.gelu(self.local_projection(
            self.local_pool(hidden, mask))))
        relations = relation_features(global_vector, local_vector)
        gate = torch.sigmoid(self.gate_output(F.gelu(self.gate_hidden(relations))))
        rho = torch.sigmoid(self.residual_scale_logit)
        fused = self.fusion_norm(global_vector + rho * gate *
            (local_vector - global_vector))
        return fused, global_vector, local_vector, gate, rho, outputs

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        if attention_mask is None:
            raise ValueError('attention_mask is required')
        kwargs.pop('num_items_in_batch', None)
        fused, global_vector, local_vector, _, _, _ = self.compute_features(
            input_ids, attention_mask, **kwargs)
        logits = self.classifier(self.dropout(fused))
        loss = None
        if labels is not None:
            per_sample_ce = F.cross_entropy(logits, labels,
                weight=self.class_weight_tensor.to(logits.dtype), reduction='none')
            true_probability = logits.softmax(dim=-1).gather(
                1, labels[:, None]).squeeze(1)
            main_loss = (((1.0 - true_probability).clamp_min(0) **
                self.config.focal_gamma) * per_sample_ce).mean()
            relations = relation_features(global_vector, local_vector)
            toxic_logit = self.toxic_head(relations).squeeze(-1)
            target_logit = self.target_head(relations).squeeze(-1)
            toxic_truth = labels.ne(0).to(logits.dtype)
            toxic_loss = F.binary_cross_entropy_with_logits(toxic_logit, toxic_truth,
                pos_weight=self.toxic_pos_weight_tensor.to(logits.dtype))
            toxic_mask = labels.ne(0)
            if toxic_mask.any():
                target_truth = labels.eq(2).to(logits.dtype)
                target_loss = F.binary_cross_entropy_with_logits(
                    target_logit[toxic_mask], target_truth[toxic_mask])
            else:
                target_loss = logits.sum() * 0.0
            loss = (main_loss + self.config.aux_toxic_weight * toxic_loss +
                self.config.aux_target_weight * target_loss)
        return SequenceClassifierOutput(loss=loss, logits=logits)

    @torch.no_grad()
    def gate_diagnostics(self, input_ids, attention_mask, **kwargs):
        _, _, _, gate, rho, _ = self.compute_features(
            input_ids, attention_mask, **kwargs)
        return gate, rho, self.layer_weights.softmax(dim=0)

print('Custom Optimized GATED v4 classes: ready')


In [ ]:
# Match NB00 text with NB03 predictions and build six balanced quick tests.
predictions = pd.read_parquet(predictions_path).reset_index(drop=True)
required_prediction_columns = {
    'y_true', 'y_pred_raw', 'p_clean', 'p_offensive', 'p_hate'
}
if not required_prediction_columns <= set(predictions.columns):
    raise ValueError(
        f'Prediction artifact is missing {required_prediction_columns - set(predictions.columns)}'
    )
if len(validation_frame) != len(predictions):
    raise RuntimeError('NB00 validation rows and NB03 predictions have different lengths.')
truth_from_data = validation_frame['labels'].astype(int).to_numpy()
truth_from_predictions = predictions['y_true'].astype(int).to_numpy()
if not np.array_equal(truth_from_data, truth_from_predictions):
    raise RuntimeError('NB00/NB03 validation label order mismatch; refusing to join.')

sample_frame = validation_frame.copy()
sample_frame['y_pred'] = predictions['y_pred_raw'].astype(int)
for column in ('p_clean', 'p_offensive', 'p_hate'):
    sample_frame[column] = predictions[column].astype(float)
sample_frame['confidence'] = sample_frame[
    ['p_clean', 'p_offensive', 'p_hate']
].max(axis=1)
sample_frame['text'] = sample_frame['text'].fillna('').astype(str).str.strip()
correct = sample_frame[
    (sample_frame['labels'].astype(int) == sample_frame['y_pred']) &
    sample_frame['text'].ne('')
].drop_duplicates(subset=['text']).copy()

demo_examples = []
for label_id, amount in SAMPLES_PER_CLASS.items():
    candidates = correct[correct['labels'].astype(int) == label_id].copy()
    displayable = candidates[candidates['text'].str.len().between(2, 280)]
    if len(displayable) >= amount:
        candidates = displayable
    selected = candidates.sort_values(
        ['confidence', 'sample_id'], ascending=[False, True]
    ).head(amount)
    if len(selected) != amount:
        raise RuntimeError(f'Not enough correct samples for {ID2LABEL[label_id]}')
    for _, row in selected.iterrows():
        demo_examples.append({
            'sample_id': str(row['sample_id']),
            'text': str(row['text']),
            'expected_label': ID2LABEL[label_id],
            'predicted_label': ID2LABEL[int(row['y_pred'])],
            'confidence': float(row['confidence']),
            'source_split': 'validation',
        })

if len(demo_examples) != 6:
    raise RuntimeError(f'Expected six demo examples, found {len(demo_examples)}')
print(pd.DataFrame(demo_examples)[
    ['sample_id', 'expected_label', 'predicted_label', 'confidence']
].to_string(index=False))


In [ ]:
# Load the complete offline inference artifact.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = OptimizedGateViSoBERT.from_pretrained(
    model_dir, local_files_only=True
).to(device).eval()

def render_empty_result():
    return '''
    <section class="result-shell is-empty" aria-live="polite">
      <p class="result-title">Chưa có dự đoán</p>
      <p class="result-note">Chọn một sample hoặc nhập bình luận rồi bấm Phân tích.</p>
    </section>
    '''

def render_prediction(scores, predicted_label, latency_ms):
    rows = []
    for label in ('CLEAN', 'OFFENSIVE', 'HATE'):
        probability = float(scores[label])
        rows.append(f'''
        <div class="prob-row">
          <div class="prob-meta"><span>{label}</span><strong>{probability:.1%}</strong></div>
          <div class="prob-track" role="progressbar" aria-label="{label}"
               aria-valuenow="{probability * 100:.1f}" aria-valuemin="0" aria-valuemax="100">
            <span style="width:{probability * 100:.1f}%"></span>
          </div>
        </div>''')
    return f'''
    <section class="result-shell" aria-live="polite">
      <p class="result-kicker">Kết quả raw softmax</p>
      <div class="prediction-line">{predicted_label}</div>
      {''.join(rows)}
      <div class="result-foot">
        <span>Seed {best_seed}</span><span>{latency_ms:.1f} ms</span>
      </div>
    </section>
    '''

def predict_text(text):
    value = str(text or '').strip()
    if not value:
        raise ValueError('Vui lòng nhập một câu tiếng Việt cần phân tích.')
    encoded = tokenizer(
        value, truncation=True, max_length=MAX_LENGTH, padding=True,
        return_tensors='pt', return_token_type_ids=False,
    )
    encoded = {key: tensor.to(device) for key, tensor in encoded.items()}
    if device.type == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits.float(), dim=-1)[0].cpu().numpy()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - started) * 1000.0
    predicted_id = int(probabilities.argmax())
    scores = {ID2LABEL[index]: float(probabilities[index]) for index in range(3)}
    return render_prediction(scores, ID2LABEL[predicted_id], latency_ms)

# Warm up once so the first user action is not dominated by CUDA initialization.
_ = predict_text('Đây là câu kiểm tra khởi động mô hình.')
print({'device': str(device), 'model_ready': True, 'selected_seed': best_seed})


In [ ]:
import gradio as gr

PIXEL_CSS = r'''
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&family=Pixelify+Sans:wght@500;600&display=swap');

:root {
  --avocado-ground: #edf6d5;
  --avocado-soft: #f7fbe9;
  --avocado-active: #dff0ad;
  --avocado-data: #86ad28;
  --avocado-ink: #1f2915;
  --muted: #5d6753;
  --surface: #ffffff;
  --border: #dce8ba;
  --control: #bdcc95;
}
body, .gradio-container {
  font-family: 'IBM Plex Sans', system-ui, sans-serif !important;
  color: var(--avocado-ink);
}
body { background: var(--avocado-ground) !important; }
.gradio-container {
  max-width: 1180px !important;
  margin: 8px auto !important;
  padding: 24px !important;
  background: var(--surface) !important;
  border: 1px solid var(--border) !important;
  border-radius: 16px !important;
  box-shadow: none !important;
}
#pixel-header {
  display: grid;
  grid-template-columns: 58px 1fr;
  gap: 16px;
  align-items: center;
  margin-bottom: 18px;
}
.pixel-mark {
  width: 56px; height: 56px;
  background: var(--avocado-data);
  border: 4px solid var(--avocado-ink);
  clip-path: polygon(12px 0, 44px 0, 44px 4px, 52px 4px, 52px 12px, 56px 12px, 56px 44px, 52px 44px, 52px 52px, 44px 52px, 44px 56px, 12px 56px, 12px 52px, 4px 52px, 4px 44px, 0 44px, 0 12px, 4px 12px, 4px 4px, 12px 4px);
  position: relative;
}
.pixel-mark::before, .pixel-mark::after {
  content: ''; position: absolute; background: var(--surface); width: 8px; height: 8px; top: 14px;
}
.pixel-mark::before { left: 10px; } .pixel-mark::after { right: 10px; }
.pixel-title {
  margin: 0; font-family: 'Pixelify Sans', 'IBM Plex Sans', sans-serif;
  font-size: 30px; line-height: 1; font-weight: 600; color: var(--avocado-ink);
}
.pixel-subtitle { margin: 7px 0 0; color: var(--muted); font-size: 14px; line-height: 1.45; }
.model-strip {
  display: grid; grid-template-columns: repeat(4, 1fr);
  border: 1px solid var(--border); border-radius: 8px; overflow: hidden; margin-bottom: 20px;
}
.model-fact { padding: 13px 16px; background: var(--avocado-soft); border-right: 1px solid var(--border); }
.model-fact:last-child { border-right: 0; }
.model-fact span { display: block; color: var(--muted); font-size: 12px; }
.model-fact strong { display: block; margin-top: 3px; font-size: 16px; font-weight: 600; font-variant-numeric: tabular-nums; }
.section-heading { font-size: 16px; font-weight: 600; margin: 0 0 6px; }
.section-note { color: var(--muted); font-size: 13px; margin: 0 0 12px; }
.gradio-container .block { box-shadow: none !important; }
.gradio-container textarea, .gradio-container input {
  border-color: var(--control) !important; background: #fff !important;
}
.gradio-container textarea:focus, .gradio-container input:focus {
  border-color: var(--avocado-data) !important; box-shadow: 0 0 0 2px var(--avocado-active) !important;
}
#analyze-button {
  background: var(--avocado-ink) !important; color: #fff !important;
  border: 2px solid var(--avocado-ink) !important; border-radius: 6px !important;
  clip-path: polygon(6px 0, 100% 0, 100% calc(100% - 6px), calc(100% - 6px) 100%, 0 100%, 0 6px);
  transition: transform 100ms steps(2, end), background 100ms linear;
}
#analyze-button:hover { background: #354225 !important; transform: translateY(-2px); }
#analyze-button:active { transform: translateY(1px); }
#clear-button { background: #fff !important; border: 1px solid var(--control) !important; color: var(--avocado-ink) !important; }
.sample-button {
  min-height: 88px !important; text-align: left !important; white-space: normal !important;
  justify-content: flex-start !important; padding: 12px 14px !important;
  background: var(--avocado-soft) !important; color: var(--avocado-ink) !important;
  border: 1px solid var(--border) !important; border-radius: 6px !important;
  clip-path: polygon(6px 0, 100% 0, 100% calc(100% - 6px), calc(100% - 6px) 100%, 0 100%, 0 6px);
  transition: background 100ms linear, transform 100ms steps(2, end);
}
.sample-button:hover { background: var(--avocado-active) !important; transform: translateY(-2px); }
.result-shell {
  min-height: 278px; padding: 18px; background: var(--avocado-soft);
  border: 1px solid var(--border); border-radius: 8px;
}
.result-shell.is-empty { display: flex; min-height: 278px; flex-direction: column; justify-content: center; }
.result-title { font-weight: 600; font-size: 16px; margin: 0; }
.result-note, .result-kicker { color: var(--muted); font-size: 12px; margin: 5px 0 0; }
.prediction-line {
  margin: 4px 0 16px; font-family: 'Pixelify Sans', 'IBM Plex Sans', sans-serif;
  font-size: 28px; font-weight: 600; color: var(--avocado-ink);
}
.prob-row { margin-top: 11px; }
.prob-meta { display: flex; justify-content: space-between; font-size: 12px; }
.prob-meta strong { font-family: 'IBM Plex Mono', monospace; font-weight: 500; font-variant-numeric: tabular-nums; }
.prob-track { height: 9px; margin-top: 5px; background: #e5ead8; overflow: hidden; border-radius: 2px; }
.prob-track span { display: block; height: 100%; background: var(--avocado-data); transition: width 220ms steps(8, end); }
.result-foot { display: flex; justify-content: space-between; margin-top: 18px; color: var(--muted); font-size: 12px; font-family: 'IBM Plex Mono', monospace; }
.privacy-note {
  margin-top: 18px; padding: 10px 12px; background: #fffbea;
  border: 1px solid #eadb98; border-radius: 6px; color: #6e5810; font-size: 12px;
}
@media (max-width: 760px) {
  body { background: #fff !important; }
  .gradio-container { margin: 0 !important; padding: 16px !important; border: 0 !important; border-radius: 0 !important; }
  #pixel-header { grid-template-columns: 44px 1fr; gap: 12px; }
  .pixel-mark { width: 44px; height: 44px; transform: scale(.78); transform-origin: left center; }
  .pixel-title { font-size: 24px; }
  .model-strip { grid-template-columns: repeat(2, 1fr); }
  .model-fact:nth-child(2) { border-right: 0; }
  .model-fact:nth-child(-n+2) { border-bottom: 1px solid var(--border); }
  .sample-button { min-height: 74px !important; }
}
@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { transition: none !important; animation: none !important; }
}
'''

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.lime,
    neutral_hue=gr.themes.colors.gray,
    radius_size=gr.themes.sizes.radius_sm,
    font=['IBM Plex Sans', 'system-ui', 'sans-serif'],
).set(
    body_background_fill='#edf6d5',
    block_background_fill='#ffffff',
    block_border_color='#dce8ba',
    button_primary_background_fill='#1f2915',
    button_primary_background_fill_hover='#354225',
)

def sample_button_text(index, item):
    preview = ' '.join(item['text'].split())
    if len(preview) > 82:
        preview = preview[:79].rstrip() + '…'
    return f"{index}. {item['expected_label']}  ·  {preview}"

header_html = '''
<header id="pixel-header">
  <div class="pixel-mark" aria-hidden="true"></div>
  <div>
    <h1 class="pixel-title">ViTox Pixel Lab</h1>
    <p class="pixel-subtitle">Phân loại bình luận tiếng Việt thành CLEAN, OFFENSIVE hoặc HATE bằng Optimized GATED v4.</p>
  </div>
</header>
'''
model_strip_html = f'''
<section class="model-strip" aria-label="Thông tin model">
  <div class="model-fact"><span>Checkpoint</span><strong>Seed {best_seed}</strong></div>
  <div class="model-fact"><span>Validation Macro-F1</span><strong>{best_validation_macro_f1:.4f}</strong></div>
  <div class="model-fact"><span>Artifact</span><strong>{html.escape(artifact_origin)}</strong></div>
  <div class="model-fact"><span>Accelerator</span><strong>{html.escape(str(device).upper())}</strong></div>
</section>
'''

with gr.Blocks(theme=theme, css=PIXEL_CSS, title='ViTox Pixel Lab') as demo:
    gr.HTML(header_html)
    gr.HTML(model_strip_html)

    with gr.Row(equal_height=False):
        with gr.Column(scale=7, min_width=340):
            gr.HTML('<h2 class="section-heading">Phân tích bình luận</h2><p class="section-note">Nhập một câu mới hoặc chọn quick test phía dưới.</p>')
            text_input = gr.Textbox(
                label='Nội dung tiếng Việt',
                placeholder='Nhập bình luận cần kiểm tra…',
                lines=7, max_lines=12, show_copy_button=True,
            )
            with gr.Row():
                analyze_button = gr.Button('Phân tích', variant='primary', elem_id='analyze-button')
                clear_button = gr.Button('Xóa', elem_id='clear-button')
        with gr.Column(scale=5, min_width=300):
            gr.HTML('<h2 class="section-heading">Xác suất dự đoán</h2><p class="section-note">Quy tắc chính: raw softmax + argmax.</p>')
            result_output = gr.HTML(render_empty_result())

    gr.HTML('<h2 class="section-heading" style="margin-top:22px">Sáu quick tests đã dự đoán đúng</h2><p class="section-note">Hai mẫu cho mỗi lớp, lấy từ validation và xếp theo confidence. Nhãn trên nút là ground truth.</p>')
    sample_buttons = []
    for row_start in (0, 3):
        with gr.Row(equal_height=True):
            for index in range(row_start, row_start + 3):
                item = demo_examples[index]
                button = gr.Button(
                    sample_button_text(index + 1, item),
                    elem_classes=['sample-button'],
                )
                sample_buttons.append(button)

    gr.HTML('''<div class="privacy-note">Các quick tests là ví dụ được tuyển chọn từ những hàng validation model đã đoán đúng; chúng không đại diện cho độ chính xác toàn tập. Không bật public share nếu chưa có quyền phân phối nguyên văn ViHSD.</div>''')

    analyze_button.click(
        fn=predict_text, inputs=text_input, outputs=result_output,
        api_name='predict', show_progress='minimal',
    )
    text_input.submit(
        fn=predict_text, inputs=text_input, outputs=result_output,
        show_progress='minimal',
    )
    clear_button.click(
        fn=lambda: ('', render_empty_result()),
        inputs=None, outputs=[text_input, result_output],
    )
    for button, item in zip(sample_buttons, demo_examples):
        sample_state = gr.State(item['text'])
        button.click(
            fn=lambda value: value, inputs=sample_state, outputs=text_input,
            show_progress='hidden',
        ).then(
            fn=predict_text, inputs=text_input, outputs=result_output,
            show_progress='minimal',
        )

print('Gradio interface: ready')


In [ ]:
# On Kaggle, the app is shown inline. Public sharing stays opt-in because examples are gated data.
demo.queue(default_concurrency_limit=2).launch(
    share=SHARE_APP, inline=True, show_error=True, debug=True,
)
